# 18.2 实验追踪与模型注册 (Experiment Tracking & Model Registry)

> 🕐 预估学习时间：35分钟

在 LLM 训练过程中，一次实验往往涉及数十个超参数、上百次迭代和多个模型版本。没有系统化的追踪机制，复现实验、对比方案、回滚故障几乎不可能。本节用纯 Python 模拟 MLflow / Weights & Biases 的核心能力，帮助你理解实验追踪与模型注册的工程实践。

本节涵盖：
- 实验追踪核心概念（参数、指标、版本、环境）
- ExperimentTracker 实现：记录、对比、持久化
- 模型注册表：版本管理、阶段晋升、回滚
- 超参数搜索追踪：Grid / Random Search
- 多实验对比与可视化

## 1. 实验追踪核心概念

**为什么需要实验追踪？**
- **可复现性**：相同参数 + 相同环境 → 相同结果
- **可对比性**：在统一视角下评估不同方案
- **可审计性**：追溯任意生产模型的训练来源
- **团队协作**：避免重复实验，共享知识

**需要追踪的内容**：

| 类别 | 示例 |
|------|------|
| 超参数 | learning_rate, batch_size, hidden_dim |
| 指标 | loss, accuracy, perplexity, BLEU |
| 模型版本 | checkpoint 路径、git commit、参数哈希 |
| 环境 | Python 版本、CUDA 版本、GPU 型号 |
| 数据 | 数据集版本、采样比例、预处理流水线 |
| 训练日志 | 每 epoch 的 loss 曲线、梯度范数 |

**主流工具对比**：
- **MLflow**：开源、自托管、组件齐全
- **Weights & Biases**：SaaS、可视化强、协作好
- **TensorBoard**：轻量、与 TF/PyTorch 深度集成
- **ClearML / Neptune**：企业级、自动化追踪

In [ ]:
import torch
import time
from collections import defaultdict

torch.manual_seed(42)

class ExperimentTracker:
    def __init__(self, name):
        self.name = name
        self.experiments = {}

    def create_experiment(self, params):
        exp_id = f'exp_{len(self.experiments) + 1:03d}'
        self.experiments[exp_id] = {
            'id': exp_id,
            'params': dict(params),
            'metrics': defaultdict(list),
            'status': 'running',
            'created_at': time.time(),
        }
        return exp_id

    def log_metric(self, exp_id, key, value, step=None):
        history = self.experiments[exp_id]['metrics'][key]
        s = step if step is not None else len(history)
        history.append((s, value))

    def finish(self, exp_id, status='completed'):
        self.experiments[exp_id]['status'] = status

    def get_best_metric(self, exp_id, key, mode='min'):
        values = [v for _, v in self.experiments[exp_id]['metrics'][key]]
        if not values:
            return None
        return min(values) if mode == 'min' else max(values)

    def summary(self):
        print(f'\n=== {self.name} Summary ===')
        header = f'{"ID":<10}{"Status":<12}{"lr":<10}{"batch":<8}{"best_loss":<12}{"best_acc":<12}'
        print(header)
        print('-' * 64)
        for exp_id, exp in self.experiments.items():
            best_loss = self.get_best_metric(exp_id, 'loss', 'min')
            best_acc = self.get_best_metric(exp_id, 'accuracy', 'max')
            lr = exp['params'].get('lr', '-')
            bs = exp['params'].get('batch_size', '-')
            loss_str = f'{best_loss:.4f}' if best_loss is not None else '-'
            acc_str = f'{best_acc:.4f}' if best_acc is not None else '-'
            status = exp['status']
            print(f'{exp_id:<10}{status:<12}{lr:<10}{bs:<8}{loss_str:<12}{acc_str:<12}')

tracker = ExperimentTracker('LLM-Training')

def simulate_training(tracker, params, n_epochs=5):
    exp_id = tracker.create_experiment(params)
    torch.manual_seed(42)
    base_loss = 2.5
    lr = params['lr']
    decay = 0.85 + 0.1 * (1 - min(lr, 1.0))
    for epoch in range(n_epochs):
        loss = base_loss * (decay ** epoch) + 0.05 * torch.randn(1).item()
        acc = 1 - loss / 3 + 0.02 * torch.randn(1).item()
        tracker.log_metric(exp_id, 'loss', loss, step=epoch)
        tracker.log_metric(exp_id, 'accuracy', acc, step=epoch)
    tracker.finish(exp_id)
    return exp_id

configs = [
    {'lr': 0.1, 'batch_size': 32},
    {'lr': 0.01, 'batch_size': 64},
    {'lr': 0.001, 'batch_size': 128},
]

print('=== ExperimentTracker Demo ===')
for cfg in configs:
    eid = simulate_training(tracker, cfg)
    lr = cfg['lr']
    bs = cfg['batch_size']
    print(f'  Finished {eid}: lr={lr}, batch_size={bs}')

tracker.summary()
print(f'\nKey: ExperimentTracker captures params + per-step metrics for reproducible comparison.')
print(f'Each experiment gets a unique ID, enabling later lookup and rollback.')

## 2. 模型注册表 (Model Registry)

**模型注册表的核心职责**：
- **版本管理**：每个 checkpoint 对应一个不可变版本
- **阶段管理**：None → Staging → Production → Archived
- **元数据存储**：训练数据、指标、git commit、参数哈希
- **回滚能力**：快速从 Production 回滚到上一版本

**典型工作流**：
```
训练完成 → register(version=1.0, stage=None)
         → 验证通过 → promote(stage=Staging)
         → 灰度通过 → promote(stage=Production)
         → 发现问题 → rollback(to=previous_production)
         → 旧版本 → archive(stage=Archived)
```

**MLflow Model Registry 阶段**：
- `None`：刚注册，未评估
- `Staging`：预发布，灰度验证
- `Production`：线上服务
- `Archived`：归档，保留历史

In [ ]:
class ModelRegistry:
    STAGES = ['None', 'Staging', 'Production', 'Archived']

    def __init__(self):
        self.models = defaultdict(list)

    def register(self, name, version, path, metrics=None, params=None):
        entry = {
            'name': name,
            'version': version,
            'path': path,
            'stage': 'None',
            'metrics': metrics or {},
            'params': params or {},
            'registered_at': time.time(),
        }
        self.models[name].append(entry)
        return entry

    def _find(self, name, version):
        for entry in self.models[name]:
            if entry['version'] == version:
                return entry
        return None

    def transition_stage(self, name, version, target_stage):
        if target_stage not in self.STAGES:
            raise ValueError(f'Unknown stage: {target_stage}')
        entry = self._find(name, version)
        if entry is None:
            raise KeyError(f'{name}@{version} not found')
        if target_stage == 'Production':
            for other in self.models[name]:
                if other is not entry and other['stage'] == 'Production':
                    other['stage'] = 'Archived'
        entry['stage'] = target_stage
        return entry

    def rollback(self, name):
        versions = self.models[name]
        prod = [v for v in versions if v['stage'] == 'Production']
        archived = [v for v in versions if v['stage'] == 'Archived']
        if not prod or not archived:
            return None
        current = prod[0]
        previous = max(archived, key=lambda v: v['version'])
        current['stage'] = 'Archived'
        previous['stage'] = 'Production'
        return previous

    def list_versions(self, name):
        return sorted(self.models[name], key=lambda v: v['version'])

    def get_production(self, name):
        for entry in self.models[name]:
            if entry['stage'] == 'Production':
                return entry
        return None

registry = ModelRegistry()
registry.register('llama-chat', '1.0', '/models/llama-1.0', {'loss': 1.8, 'acc': 0.72}, {'lr': 0.1})
registry.register('llama-chat', '1.1', '/models/llama-1.1', {'loss': 1.5, 'acc': 0.78}, {'lr': 0.05})
registry.register('llama-chat', '1.2', '/models/llama-1.2', {'loss': 1.3, 'acc': 0.82}, {'lr': 0.03})

registry.transition_stage('llama-chat', '1.0', 'Production')
registry.transition_stage('llama-chat', '1.1', 'Staging')
registry.transition_stage('llama-chat', '1.2', 'Staging')

print('=== Model Registry Status ===')
header = f'{"Name":<14}{"Version":<10}{"Stage":<12}{"loss":<8}{"acc":<8}{"path"}'
print(header)
print('-' * 70)
for v in registry.list_versions('llama-chat'):
    name = v['name']
    ver = v['version']
    stage = v['stage']
    loss = v['metrics']['loss']
    acc = v['metrics']['acc']
    path = v['path']
    print(f'{name:<14}{ver:<10}{stage:<12}{loss:<8}{acc:<8}{path}')

print('\n--- Promote 1.2 to Production ---')
registry.transition_stage('llama-chat', '1.2', 'Production')
prod = registry.get_production('llama-chat')
prod_ver = prod['version'] if prod else 'none'
print(f'Production now: v{prod_ver}')

print('\n--- Rollback to previous Production ---')
rolled = registry.rollback('llama-chat')
if rolled:
    rolled_ver = rolled['version']
    print(f'Rolled back to: v{rolled_ver}')

print('\n--- Final Registry State ---')
for v in registry.list_versions('llama-chat'):
    ver = v['version']
    stage = v['stage']
    print(f'  v{ver}: stage={stage}')

print(f'\nKey: ModelRegistry enforces one Production version at a time.')
print(f'Rollback swaps Production with the latest Archived version for fast recovery.')

## 3. 超参数搜索追踪

**搜索策略对比**：

| 策略 | 优点 | 缺点 |
|------|------|------|
| **Grid Search** | 简单、可复现 | 维度灾难、效率低 |
| **Random Search** | 高维友好、易并行 | 不保证最优 |
| **Bayesian Optimization** | 样本高效、自适应 | 实现复杂、串行 |
| **Hyperband / ASHA** | 早停节省资源 | 需要可中断训练 |

**追踪要点**：
- 每个 trial 都是一个独立 Experiment
- 记录完整参数组合 + 最终指标
- 保留中间日志以便分析训练动态
- 标记 best trial 便于后续 promote

In [ ]:
import itertools
import random

class HyperparameterSearch:
    def __init__(self, tracker, mode='grid'):
        self.tracker = tracker
        self.mode = mode
        self.trials = []

    def _simulate(self, params, n_epochs=5):
        torch.manual_seed(42)
        base_loss = 2.5
        lr = params['lr']
        bs = params['batch_size']
        decay = 0.7 + 0.2 * (1.0 / (1.0 + lr * 100)) + 0.05 * (bs / 128.0)
        exp_id = self.tracker.create_experiment(params)
        for epoch in range(n_epochs):
            loss = base_loss * (decay ** epoch) + 0.03 * torch.randn(1).item()
            acc = 1 - loss / 3 + 0.01 * torch.randn(1).item()
            self.tracker.log_metric(exp_id, 'loss', loss, step=epoch)
            self.tracker.log_metric(exp_id, 'accuracy', acc, step=epoch)
        self.tracker.finish(exp_id)
        best_loss = self.tracker.get_best_metric(exp_id, 'loss', 'min')
        best_acc = self.tracker.get_best_metric(exp_id, 'accuracy', 'max')
        return {'exp_id': exp_id, 'params': params, 'best_loss': best_loss, 'best_acc': best_acc}

    def grid_search(self, param_grid, n_epochs=5):
        keys = list(param_grid.keys())
        for values in itertools.product(*[param_grid[k] for k in keys]):
            params = dict(zip(keys, values))
            result = self._simulate(params, n_epochs)
            self.trials.append(result)
        return self.best()

    def random_search(self, param_distributions, n_trials=10, n_epochs=5):
        for _ in range(n_trials):
            params = {k: random.choice(v) for k, v in param_distributions.items()}
            result = self._simulate(params, n_epochs)
            self.trials.append(result)
        return self.best()

    def best(self, metric='best_loss', mode='min'):
        if not self.trials:
            return None
        if mode == 'min':
            return min(self.trials, key=lambda t: t[metric])
        return max(self.trials, key=lambda t: t[metric])

    def report(self):
        mode_name = self.mode.capitalize()
        n_trials = len(self.trials)
        print(f'\n=== {mode_name} Search Results ({n_trials} trials) ===')
        header = f'{"Trial":<8}{"lr":<10}{"batch_size":<14}{"best_loss":<14}{"best_acc":<14}'
        print(header)
        print('-' * 60)
        for i, t in enumerate(self.trials, 1):
            lr = t['params']['lr']
            bs = t['params']['batch_size']
            bl = t['best_loss']
            ba = t['best_acc']
            print(f'{i:<8}{lr:<10}{bs:<14}{bl:<14.4f}{ba:<14.4f}')
        best = self.best()
        if best:
            best_id = best['exp_id']
            best_lr = best['params']['lr']
            best_bs = best['params']['batch_size']
            best_loss = best['best_loss']
            best_acc = best['best_acc']
            print(f'\nBest trial: {best_id} | lr={best_lr}, batch_size={best_bs}')
            print(f'  best_loss={best_loss:.4f}, best_acc={best_acc:.4f}')

search_tracker = ExperimentTracker('HPO-Grid')
search = HyperparameterSearch(search_tracker, mode='grid')
grid = {
    'lr': [0.1, 0.01, 0.001],
    'batch_size': [32, 64, 128],
}
search.grid_search(grid, n_epochs=4)
search.report()

n_trials = len(search.trials)
print(f'\nKey: Grid search exhaustively evaluates all combinations ({n_trials} trials).')
print(f'Tracking each trial as an Experiment enables later analysis of training dynamics.')

## 4. 实验对比与可视化

**对比维度**：
- **最终指标**：best loss / accuracy / perplexity
- **收敛速度**：达到目标指标所需 epoch 数
- **稳定性**：指标方差、梯度范数波动
- **资源消耗**：显存峰值、训练时长

**可视化技巧**：
- 学习曲线（loss vs epoch）叠加多条实验
- 参数-指标散点图发现敏感参数
- 平行坐标图展示高维参数空间
- 热力图对比 grid search 结果

In [ ]:
def compare_experiments(tracker, exp_ids):
    print('=== Experiment Comparison ===')
    header = f'{"ID":<10}{"lr":<10}{"batch":<8}{"epochs":<8}{"best_loss":<14}{"best_acc":<14}{"final_loss":<14}'
    print(header)
    print('-' * 78)
    for exp_id in exp_ids:
        exp = tracker.experiments[exp_id]
        losses = [v for _, v in exp['metrics']['loss']]
        accs = [v for _, v in exp['metrics']['accuracy']]
        best_loss = min(losses) if losses else float('nan')
        best_acc = max(accs) if accs else float('nan')
        final_loss = losses[-1] if losses else float('nan')
        lr = exp['params']['lr']
        bs = exp['params']['batch_size']
        n_epochs = len(losses)
        print(f'{exp_id:<10}{lr:<10}{bs:<8}{n_epochs:<8}{best_loss:<14.4f}{best_acc:<14.4f}{final_loss:<14.4f}')

    print('\n--- Loss Curves (text) ---')
    max_epochs = max(len(tracker.experiments[e]['metrics']['loss']) for e in exp_ids)
    header = f'{"epoch":<8}' + ''.join(f'{e:<14}' for e in exp_ids)
    print(header)
    for epoch in range(max_epochs):
        row = f'{epoch:<8}'
        for exp_id in exp_ids:
            losses = tracker.experiments[exp_id]['metrics']['loss']
            if epoch < len(losses):
                val = losses[epoch][1]
                row += f'{val:<14.4f}'
            else:
                row += f'{"-":<14}'
        print(row)

    best_overall = min(exp_ids, key=lambda e: min(v for _, v in tracker.experiments[e]['metrics']['loss']))
    best_loss_val = min(v for _, v in tracker.experiments[best_overall]['metrics']['loss'])
    print(f'\nBest overall: {best_overall} (best_loss={best_loss_val:.4f})')
    return best_overall

compare_tracker = ExperimentTracker('Comparison')
compare_configs = [
    {'lr': 0.1, 'batch_size': 32},
    {'lr': 0.01, 'batch_size': 64},
    {'lr': 0.001, 'batch_size': 128},
]
compare_ids = [simulate_training(compare_tracker, cfg, n_epochs=6) for cfg in compare_configs]
compare_experiments(compare_tracker, compare_ids)

print(f'\nKey: Side-by-side comparison reveals lr=0.1 converges fast but may be unstable.')
print(f'Lower lr gives smoother curves but needs more epochs to reach the same loss.')

## 📝 课后思考题

1. 在分布式训练中，如何确保多个 worker 记录的指标一致且不重复？
2. 如果实验追踪系统本身成为瓶颈（如每秒上千次 metric 写入），你会如何优化？
3. 模型从 Staging 晋升到 Production 之前，应该自动检查哪些质量门禁？
4. 如何设计一个实验追踪系统，使其同时支持在线实验（A/B 测试）和离线实验？